In [27]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import seaborn as sns

from functions import get_db_connection

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
import pandas as pd
from functions import get_db_connection, load_all_tables

con = get_db_connection()

# Load all tables into a dictionary
db_tables = load_all_tables(con)

# Unpack into separate DataFrames
df_sensors = db_tables["sensors"]
df_hubs = db_tables["hubs"]

df_measurements_span1 = db_tables["measurements_span1"]
df_measurements_span2 = db_tables["measurements_span2"]
df_measurements_span3 = db_tables["measurements_span3"]
df_measurements_span4 = db_tables["measurements_span4"]
df_measurements_span5 = db_tables["measurements_span5"]

con.close()

In [29]:
print(df_hubs)

         meta_uuid   sn  span                              sensors
0  185284862261152  128     1  [6, 10, 11, 17, 19, 20, 25, 27, 41]
1   44332625541024  116     2                   [2, 3, 15, 28, 42]
2   39714511164468  115     3   [1, 8, 12, 13, 14, 23, 26, 30, 43]
3  207137655863200  114     4              [4, 22, 24, 29, 31, 44]
4   97186493085600  112     5            [5, 7, 9, 16, 18, 21, 45]


In [30]:
df_sensors.sample(1).iloc[0]

sensor_id                 9
meta_uuid    97186493085600
position                  6
type                  DLeaf
color               #F09EA6
probes           [pv0, pv1]
tare_pv0            0.63307
tare_pv1            0.58392
tare_pv2                NaN
tare_pv3                NaN
Name: 8, dtype: object

In [31]:
df_measurements_span1.sample(1).iloc[0]

timestamp        2026-07-08 12:37:03
voltage                        4.206
values_6_pv0                 0.38811
values_6_pv1                0.376076
values_10_pv0               0.427712
values_10_pv1               0.482537
values_11_pv0               0.407672
values_11_pv1               0.572344
values_17_pv0               0.593198
values_17_pv1                0.42709
values_19_pv0               0.470774
values_19_pv1               0.538234
values_20_pv0               0.786263
values_20_pv1               0.600835
values_25_pv0               0.385257
values_25_pv1                0.56157
values_27_pv0               0.382203
values_27_pv1               0.400942
values_41_pv0               0.449546
values_41_pv1               0.507117
Name: 21175, dtype: object

In [32]:
# import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
# import matplotlib.ticker as ticker
# import pandas as pd
# import seaborn as sns
# from functions import (
#     find_ref_sensor,
#     save_a4_svg,
#     setup_a4_landscape_plot,
# )


# def plot_monthly_sensor_data(
#     df_measurements: pd.DataFrame,
#     df_sensors: pd.DataFrame,
#     df_hubs: pd.DataFrame,
#     sensor_id: int,
#     year: int,
#     month: int,
#     scale_factor: float = 25.0,
#     preview: bool = True,
#     save_plot: bool = False,
#     output_path: str | None = None,
# ) -> None:
#     # 0. Get main sensor metadata
#     sensor_row = df_sensors[df_sensors["sensor_id"] == sensor_id]
#     if sensor_row.empty:
#         raise ValueError(f"Sensor ID {sensor_id} not found in df_sensors.")

#     sensor_info = sensor_row.iloc[0]
#     line_color = (
#         sensor_info["color"]
#         if pd.notna(sensor_info["color"]) and sensor_info["color"]
#         else "#df77b4"
#     )
#     tare_pv0 = (
#         sensor_info["tare_pv0"] if pd.notna(sensor_info["tare_pv0"]) else 0.0
#     )
#     tare_pv1 = (
#         sensor_info["tare_pv1"] if pd.notna(sensor_info["tare_pv1"]) else 0.0
#     )

#     # Fetch reference sensor metadata
#     ref_sensor_id = find_ref_sensor(sensor_id, df_hubs, df_sensors)
#     ref_sensor_info = df_sensors[df_sensors["sensor_id"] == ref_sensor_id].iloc[0]
#     ref_tare_pv0 = (
#         ref_sensor_info["tare_pv0"] if pd.notna(ref_sensor_info["tare_pv0"]) else 0.0
#     )
#     ref_tare_pv1 = (
#         ref_sensor_info["tare_pv1"] if pd.notna(ref_sensor_info["tare_pv1"]) else 0.0
#     )

#     # 1. Set date boundaries for requested month
#     start_date = pd.Timestamp(year=year, month=month, day=1)
#     end_date = (
#         start_date
#         + pd.offsets.MonthEnd(1)
#         + pd.Timedelta(hours=23, minutes=59, seconds=59)
#     )

#     df_filtered = df_measurements[
#         (df_measurements["timestamp"] >= start_date)
#         & (df_measurements["timestamp"] <= end_date)
#     ].copy()

#     if df_filtered.empty:
#         print(f"No data available for Sensor {sensor_id} in {year}-{month:02d}.")
#         return

#     # 2. Subtract tare and scale for main and reference sensors
#     df_filtered["pv0_scaled"] = (
#         df_filtered[f"values_{sensor_id}_pv0"] - tare_pv0
#     ) * scale_factor
#     df_filtered["pv1_scaled"] = (
#         df_filtered[f"values_{sensor_id}_pv1"] - tare_pv1
#     ) * scale_factor

#     df_filtered["ref_pv0_scaled"] = (
#         df_filtered[f"values_{ref_sensor_id}_pv0"] - ref_tare_pv0
#     ) * scale_factor
#     df_filtered["ref_pv1_scaled"] = (
#         df_filtered[f"values_{ref_sensor_id}_pv1"] - ref_tare_pv1
#     ) * scale_factor

#     # 3. Initialize figure
#     fig, ax = setup_a4_landscape_plot()

#     ax.grid(True, axis="y")
#     ax.grid(False, axis="x")

#     # 4. Draw plots for main sensor
#     sns.lineplot(
#         data=df_filtered,
#         x="timestamp",
#         y="pv0_scaled",
#         ax=ax,
#         color=line_color,
#         label=f"S{sensor_id} pv0",
#         zorder=3,
#     )
#     sns.lineplot(
#         data=df_filtered,
#         x="timestamp",
#         y="pv1_scaled",
#         ax=ax,
#         color=line_color,
#         alpha=0.7,
#         linestyle="-",
#         label=f"S{sensor_id} pv1",
#         zorder=3,
#     )

#     # Draw plots for reference sensor (#000000, alpha 0.9 for pv0, alpha 0.7 for pv1)
#     sns.lineplot(
#         data=df_filtered,
#         x="timestamp",
#         y="ref_pv0_scaled",
#         ax=ax,
#         color="#000000",
#         alpha=0.7,
#         label=f"Ref (S{ref_sensor_id}) pv0",
#         zorder=3,
#     )
#     sns.lineplot(
#         data=df_filtered,
#         x="timestamp",
#         y="ref_pv1_scaled",
#         ax=ax,
#         color="#000000",
#         alpha=0.5,
#         linestyle="-",
#         label=f"Ref (S{ref_sensor_id}) pv1",
#         zorder=3,
#     )

#     # 5. Set hard plot limits (X-axis tight to month, Y-axis -0.25 to 0.25)
#     ax.set_xlim(start_date, end_date)
#     ax.set_ylim(-0.25, 0.25)

#     # 6. Set daily ticks and custom label formatter with Slovak day names
#     ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))

#     SLOVAK_DAYS = ["Po", "Ut", "St", "Št", "Pi", "So", "Ne"]

#     def custom_date_formatter(x, pos=None):
#         dt = mdates.num2date(x)
#         day_num = dt.strftime("%d")
#         if dt.weekday() == 6:  # Sunday
#             return f"{day_num} {SLOVAK_DAYS[6]}"
#         return day_num

#     ax.xaxis.set_major_formatter(ticker.FuncFormatter(custom_date_formatter))

#     # 7. Custom vertical gridlines with day-of-week alpha
#     BASE_COLOR = "#000000"
#     ALPHA_DAYS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
#     ALPHA_DAYS = [a * 0.5 for a in ALPHA_DAYS]

#     all_days = pd.date_range(
#         start=start_date.floor("D"),
#         end=end_date.floor("D"),
#         freq="D",
#     )

#     for day in all_days:
#         day_alpha = ALPHA_DAYS[day.weekday()]
#         ax.axvline(
#             x=day,
#             color=BASE_COLOR,
#             linestyle="-",
#             linewidth=0.8,
#             alpha=day_alpha,
#             zorder=1,
#         )

#     # 8. Titles and formatting
#     month_name = start_date.strftime("%B")
#     ax.set_title(
#         f"Sensor {sensor_id} & Ref Sensor {ref_sensor_id} - {month_name} {year} Scaled Tared Measurements",
#         fontsize=16,
#         pad=14,
#     )
#     ax.set_xlabel("")
#     ax.set_ylabel("Vzdialenosť [mm]", fontsize=12)
#     plt.xticks(rotation=45, ha="right", fontsize=10)

#     # 9. Handle saving and displaying inside Jupyter Notebook
#     if save_plot:
#         filename = output_path or f"sensor_{sensor_id}_{year}_{month:02d}_a4.svg"
#         save_a4_svg(fig, filename)

#     if preview:
#         plt.show()
#     else:
#         plt.close(fig)

In [42]:
from functions import plot_monthly_sensor_data

# Dictionary mapping span numbers to their respective measurement DataFrames
spans_data = {
    1: df_measurements_span1,
    2: df_measurements_span2,
    3: df_measurements_span3,
    4: df_measurements_span4,
    5: df_measurements_span5,
}

# Example: Plot sensor 26 (automatically uses span for sensor 26)
plot_monthly_sensor_data(
    measurements_by_span=spans_data,
    df_sensors=df_sensors,
    df_hubs=df_hubs,
    width=17.0,
    height=7.0,
    sensor_id=18,
    year=2026,
    month=1,
    save_plot=True
)

In [34]:
print(df_hubs)

         meta_uuid   sn  span                              sensors
0  185284862261152  128     1  [6, 10, 11, 17, 19, 20, 25, 27, 41]
1   44332625541024  116     2                   [2, 3, 15, 28, 42]
2   39714511164468  115     3   [1, 8, 12, 13, 14, 23, 26, 30, 43]
3  207137655863200  114     4              [4, 22, 24, 29, 31, 44]
4   97186493085600  112     5            [5, 7, 9, 16, 18, 21, 45]


In [35]:
df_sensors.sample(1).iloc[0]

sensor_id                26
meta_uuid    39714511164468
position                  6
type                  DLeaf
color               #F5904F
probes           [pv0, pv1]
tare_pv0            0.52416
tare_pv1            0.62974
tare_pv2                NaN
tare_pv3                NaN
Name: 25, dtype: object

In [ ]:
# Filter df_sensors for type "DLeaf" and get unique sensor IDs
dleaf_sensors = df_sensors[df_sensors["type"] == "DLeaf"]["sensor_id"].unique()

for sensor_id in dleaf_sensors:
    # Generate and save the plot
    plot_monthly_sensor_data(
        measurements_by_span=spans_data,
        df_sensors=df_sensors,
        df_hubs=df_hubs,
        width=17.0,
        height=7.0,
        sensor_id=sensor_id,
        year=2025,
        month=10,
        save_plot=True
    )
    
    # Print the formatted Typst image markup string
    print(f'#fullpage-image("images/sensor_{sensor_id}_2025_10.svg", caption: [Senzor {sensor_id} Január 2026])')

#fullpage-image("images/sensor_1_2026_10.svg", caption: [Senzor 1 Január 2026])
#fullpage-image("images/sensor_2_2026_10.svg", caption: [Senzor 2 Január 2026])
#fullpage-image("images/sensor_3_2026_10.svg", caption: [Senzor 3 Január 2026])
#fullpage-image("images/sensor_4_2026_10.svg", caption: [Senzor 4 Január 2026])
#fullpage-image("images/sensor_5_2026_10.svg", caption: [Senzor 5 Január 2026])
#fullpage-image("images/sensor_6_2026_10.svg", caption: [Senzor 6 Január 2026])
#fullpage-image("images/sensor_7_2026_10.svg", caption: [Senzor 7 Január 2026])
#fullpage-image("images/sensor_8_2026_10.svg", caption: [Senzor 8 Január 2026])
#fullpage-image("images/sensor_9_2026_10.svg", caption: [Senzor 9 Január 2026])
#fullpage-image("images/sensor_10_2026_10.svg", caption: [Senzor 10 Január 2026])
#fullpage-image("images/sensor_11_2026_10.svg", caption: [Senzor 11 Január 2026])
#fullpage-image("images/sensor_12_2026_10.svg", caption: [Senzor 12 Január 2026])
#fullpage-image("images/sensor_13_

In [37]:
find_span(3, df_hubs)

2

In [ ]:
report_build/images/sensor_1_2025_10.svg